In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import requests
import os
from typing import Dict, Optional

In [9]:
class GoldDataFetcher:
    def __init__(self):
        self.gold_symbol = "GC=F"  
        self.data_dir = "data"
        os.makedirs(self.data_dir, exist_ok=True)
    
    def fetch_gold_prices(self, period: str = "5y") -> pd.DataFrame:
        """Fetch gold price data from Yahoo Finance"""
        try:
            gold = yf.Ticker(self.gold_symbol)
            data = gold.history(period=period)
            data.index = pd.to_datetime(data.index)
            return data
        except Exception as e:
            print(f"Error fetching gold data: {e}")
            return pd.DataFrame()
    
    def fetch_economic_indicators(self) -> Dict[str, pd.DataFrame]:
        """Fetch relevant economic indicators that affect gold prices"""
        indicators = {}
        
        try:
            dxy = yf.Ticker("DX-Y.NYB")
            indicators['dxy'] = dxy.history(period="5y")
        except:
            print("Could not fetch DXY data")
        
        
        try:
            sp500 = yf.Ticker("^GSPC")
            indicators['sp500'] = sp500.history(period="5y")
        except:
            print("Could not fetch S&P 500 data")
        
        
        try:
            treasury = yf.Ticker("^TNX")
            indicators['treasury'] = treasury.history(period="5y")
        except:
            print("Could not fetch Treasury data")
        
        
        try:
            oil = yf.Ticker("CL=F")
            indicators['oil'] = oil.history(period="5y")
        except:
            print("Could not fetch Oil data")
        
        
        try:
            vix = yf.Ticker("^VIX")
            indicators['vix'] = vix.history(period="5y")
        except:
            print("Could not fetch VIX data")
        
        return indicators
    
    def combine_data(self) -> pd.DataFrame:
        """Combine gold prices with economic indicators"""
        gold_data = self.fetch_gold_prices()
        if gold_data.empty:
            raise ValueError("Could not fetch gold price data")
        
        # Make gold index timezone-naive
        gold_data.index = gold_data.index.tz_localize(None)

        combined_data = gold_data[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
        combined_data.columns = [f'gold_{col.lower()}' for col in combined_data.columns]

        indicators = self.fetch_economic_indicators()
        
        for name, data in indicators.items():
            if not data.empty:
                # Make indicator index timezone-naive too
                data.index = data.index.tz_localize(None)
                indicator_series = data['Close'].rename(f'{name}_close')
                combined_data = combined_data.join(indicator_series, how='left')

        combined_data = combined_data.fillna(method='ffill').dropna()

        combined_data.to_csv(os.path.join(self.data_dir, 'raw_gold_data.csv'))

        return combined_data


In [10]:
if __name__ == "__main__":
    fetcher = GoldDataFetcher()

    # 1. Fetch only gold prices
    gold_data = fetcher.fetch_gold_prices(period="5y")
    print("Gold Prices:")
    print(gold_data.head())

    # 2. Fetch economic indicators
    indicators = fetcher.fetch_economic_indicators()
    for name, df in indicators.items():
        print(f"\n{name.upper()} Data:")
        print(df.head())

    # 3. Combine everything into one dataset
    combined = fetcher.combine_data()
    print("\nCombined Dataset:")
    print(combined.head())

    # 4. The data is also saved to CSV automatically
    print("\nSaved combined dataset to data/raw_gold_data.csv")

Gold Prices:
                                  Open         High          Low        Close  \
Date                                                                            
2020-09-15 00:00:00-04:00  1955.800049  1968.500000  1955.800049  1956.300049   
2020-09-16 00:00:00-04:00  1961.300049  1961.400024  1956.699951  1960.199951   
2020-09-17 00:00:00-04:00  1939.000000  1946.000000  1933.699951  1940.000000   
2020-09-18 00:00:00-04:00  1950.500000  1952.099976  1949.000000  1952.099976   
2020-09-21 00:00:00-04:00  1946.199951  1946.199951  1883.400024  1901.199951   

                           Volume  Dividends  Stock Splits  
Date                                                        
2020-09-15 00:00:00-04:00      74        0.0           0.0  
2020-09-16 00:00:00-04:00     124        0.0           0.0  
2020-09-17 00:00:00-04:00      67        0.0           0.0  
2020-09-18 00:00:00-04:00      38        0.0           0.0  
2020-09-21 00:00:00-04:00      52        0.0         

C:\Users\User\AppData\Local\Temp\ipykernel_4276\1866821846.py:79: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  combined_data = combined_data.fillna(method='ffill').dropna()


In [7]:
combined.head()

,gold_open,gold_high,gold_low,gold_close,gold_volume,dxy_close,sp500_close,treasury_close,oil_close,vix_close
Date,,,,,,,,,,
